# Backpressure and Load Balancing
Companion notebook for the **Backpressure and Load Balancing** section of Chapter 7.

If work arrives faster than workers, tools, or model capacity can finish it, an unbounded queue turns a short burst into high memory use and cascading timeouts. Backpressure caps how much work is in flight at once.

In [1]:
import asyncio
import time

MAX_ACTIVE = 20
semaphore = asyncio.Semaphore(MAX_ACTIVE)


async def process_task(task: int) -> int:
    await asyncio.sleep(0.05)  # stand-in for a slow tool/model call
    return task


async def run_task(task: int) -> int:
    async with semaphore:
        return await process_task(task)

To prove the cap actually holds, track how many tasks are inside the semaphore *at the same time* while 60 tasks race to start:

In [2]:
current_active = 0
peak_active = 0


async def run_task_tracked(task: int) -> int:
    global current_active, peak_active
    async with semaphore:
        current_active += 1
        peak_active = max(peak_active, current_active)
        result = await process_task(task)
        current_active -= 1
        return result


start = time.perf_counter()
results = await asyncio.gather(*(run_task_tracked(i) for i in range(60)))
elapsed = time.perf_counter() - start

print(f"Completed {len(results)} tasks in {elapsed:.2f}s")
print(f"Peak concurrent tasks: {peak_active} (cap was MAX_ACTIVE={MAX_ACTIVE})")
assert peak_active <= MAX_ACTIVE

Completed 60 tasks in 0.19s
Peak concurrent tasks: 20 (cap was MAX_ACTIVE=20)


Backpressure decides *whether* work is accepted. **Load balancing** solves a different problem: once accepted, spread work so one worker doesn't go hot while others sit idle. Round-robin is fine for similar jobs; for uneven agent workloads (one task finishes in 2 seconds, another waits 2 minutes on a tool), least-loaded routing is safer:

In [3]:
def least_loaded_worker(worker_active_counts: dict[str, int]) -> str:
    """Route to whichever worker currently has the fewest active tasks."""
    return min(worker_active_counts, key=worker_active_counts.get)


workers = {"worker-a": 3, "worker-b": 1, "worker-c": 2}
print("Next task goes to:", least_loaded_worker(workers))

Next task goes to: worker-b
